<a href="https://colab.research.google.com/github/QUANHONGLE/CS421-emotion-prediction/blob/main/notebooks_p2/Q1_Corpus_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers
!pip install rouge-score
!pip install bert-score
!pip install sacrebleu
!pip install scikit-learn
!pip install pandas
!pip install numpy
!pip install torch

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7b6d3721c649c4c247806a57a28b450d5efc156bb06800cd4a0a9061f5d697c0
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# Load data
train_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_train.csv')
dev_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_dev.csv', on_bad_lines='skip')
test_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_test.csv', on_bad_lines='skip')

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)
print(train_df.columns.tolist())

Train: (11090, 12)
Dev: (987, 13)
Test: (2311, 9)
['id', 'article_id', 'conversation_id', 'turn_id', 'speaker', 'text', 'person_id_1', 'person_id_2', 'Emotion', 'EmotionalPolarity', 'Empathy', 'SelfDisclosure']


In [3]:
# Load SBERT model
print("Loading SBERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("SBERT loaded!")

# Compute embeddings for training data
print("Computing training embeddings...")
train_embeddings = sbert_model.encode(train_df['text'].astype(str).tolist(), show_progress_bar=True)
print("Embeddings done! Shape:", train_embeddings.shape)

# Normalize Emotion and Empathy scores to [0, 1]
emotion_min, emotion_max = train_df['Emotion'].min(), train_df['Emotion'].max()
empathy_min, empathy_max = train_df['Empathy'].min(), train_df['Empathy'].max()

train_df['Emotion_norm'] = (train_df['Emotion'] - emotion_min) / (emotion_max - emotion_min)
train_df['Empathy_norm'] = (train_df['Empathy'] - empathy_min) / (empathy_max - empathy_min)

# One-hot encode polarity (0, 1, 2, 3)
polarity_dummies = pd.get_dummies(train_df['EmotionalPolarity'], prefix='polarity')
train_df = pd.concat([train_df, polarity_dummies], axis=1)

print("Normalization done!")
print("Emotion range:", train_df['Emotion_norm'].min(), "-", train_df['Emotion_norm'].max())
print("Empathy range:", train_df['Empathy_norm'].min(), "-", train_df['Empathy_norm'].max())
print("Polarity columns:", polarity_dummies.columns.tolist())

Loading SBERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT loaded!
Computing training embeddings...


Batches:   0%|          | 0/347 [00:00<?, ?it/s]

Embeddings done! Shape: (11090, 384)
Normalization done!
Emotion range: 0.0 - 1.0
Empathy range: 0.0 - 1.0
Polarity columns: ['polarity_0', 'polarity_1', 'polarity_2', 'polarity_3']


In [4]:
def calculate_similarity(query_embedding, query_emotion, query_empathy, query_polarity,
                          train_embeddings, train_emotion, train_empathy, train_polarity,
                          w1=0.6, w2=0.15, w3=0.15, w4=0.1):
    # Text similarity - cosine similarity between embeddings
    s_text = cosine_similarity([query_embedding], train_embeddings)[0]

    # Emotion similarity
    s_emotion = 1 - np.abs(train_emotion - query_emotion)

    # Empathy similarity
    s_empathy = 1 - np.abs(train_empathy - query_empathy)

    # Polarity similarity
    s_polarity = (train_polarity == query_polarity).astype(float)

    # Combined score
    s_total = w1 * s_text + w2 * s_emotion + w3 * s_empathy + w4 * s_polarity

    return s_total

# Test it works
sample_embedding = train_embeddings[0]
sample_emotion = train_df['Emotion_norm'].values[0]
sample_empathy = train_df['Empathy_norm'].values[0]
sample_polarity = train_df['EmotionalPolarity'].values[0]

scores = calculate_similarity(
    sample_embedding, sample_emotion, sample_empathy, sample_polarity,
    train_embeddings,
    train_df['Emotion_norm'].values,
    train_df['Empathy_norm'].values,
    train_df['EmotionalPolarity'].values
)

print("Similarity scores shape:", scores.shape)
print("Max score:", scores.max())
print("Min score:", scores.min())
print("Best match index:", scores.argmax())
print("Best match text:", train_df['text'].values[scores.argmax()])

Similarity scores shape: (11090,)
Max score: 1.000000023841858
Min score: 0.09576375005766746
Best match index: 0
Best match text: what did you think about this article


In [6]:
def get_query_embedding(conversation_history, sbert_model):
    # Concatenate all utterances in the conversation history
    query_text = ' '.join(conversation_history)
    return sbert_model.encode(query_text)

def generate_next_utterance(conversation_history, query_emotion, query_empathy, query_polarity,
                             train_embeddings, train_df, sbert_model, used_indices=set()):
    # Get embedding for conversation history
    query_embedding = get_query_embedding(conversation_history, sbert_model)

    # Normalize query emotion and empathy using training min/max
    query_emotion_norm = (query_emotion - emotion_min) / (emotion_max - emotion_min)
    query_empathy_norm = (query_empathy - empathy_min) / (empathy_max - empathy_min)

    # Calculate similarity scores
    scores = calculate_similarity(
        query_embedding, query_emotion_norm, query_empathy_norm, query_polarity,
        train_embeddings,
        train_df['Emotion_norm'].values,
        train_df['Empathy_norm'].values,
        train_df['EmotionalPolarity'].values
    )

    # Mask already used sentences to avoid repetition
    scores[list(used_indices)] = -1

    # Get best match
    best_idx = scores.argmax()
    used_indices.add(best_idx)

    return train_df['text'].values[best_idx], best_idx

def generate_conversation(conv_history, emotion_targets, empathy_targets, polarity_targets,
                           train_embeddings, train_df, sbert_model, n_turns=5):
    generated = []
    used_indices = set()

    for i in range(n_turns):
        utterance, idx = generate_next_utterance(
            conv_history,
            emotion_targets[i],
            empathy_targets[i],
            polarity_targets[i],
            train_embeddings,
            train_df,
            sbert_model,
            used_indices
        )
        generated.append(utterance)
        conv_history.append(utterance)  # add generated utterance to history

    return generated

print("Generation functions defined!")

Generation functions defined!


In [7]:
def get_target_values(df, conv_id, turn_start=6, n_turns=5):
    conv = df[df['conversation_id'] == conv_id].sort_values('turn_id')
    targets = conv[conv['turn_id'] >= turn_start].head(n_turns)

    if len(targets) == 0:
        # if no turns available use average of conversation
        emotion_targets = [conv['Emotion'].mean()] * n_turns
        empathy_targets = [conv['Empathy'].mean()] * n_turns
        polarity_targets = [conv['EmotionalPolarity'].mode()[0]] * n_turns
    else:
        emotion_targets = targets['Emotion'].tolist()
        empathy_targets = targets['Empathy'].tolist()
        polarity_targets = targets['EmotionalPolarity'].tolist()
        # pad if less than n_turns
        while len(emotion_targets) < n_turns:
            emotion_targets.append(emotion_targets[-1])
            empathy_targets.append(empathy_targets[-1])
            polarity_targets.append(polarity_targets[-1])

    return emotion_targets, empathy_targets, polarity_targets

print("Generating for dev set...")
dev_results = []

for conv_id in dev_df['conversation_id'].unique():
    conv = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')

    # Get first 5 turns as history
    history = conv[conv['turn_id'] <= 5]['text'].astype(str).tolist()
    if len(history) == 0:
        continue

    # Get target emotion/empathy/polarity for turns 6-10
    emotion_targets, empathy_targets, polarity_targets = get_target_values(dev_df, conv_id)

    # Generate next 5 turns
    generated = generate_conversation(
        history.copy(),
        emotion_targets,
        empathy_targets,
        polarity_targets,
        train_embeddings,
        train_df,
        sbert_model
    )

    for turn_num, utterance in enumerate(generated, start=6):
        dev_results.append({
            'conversation_id': conv_id,
            'turn_number': turn_num,
            'generated_response': utterance
        })

dev_results_df = pd.DataFrame(dev_results)
print("Dev generation done!")
print(f"Total generated turns: {len(dev_results_df)}")
print(dev_results_df.head(10))

Generating for dev set...
Dev generation done!
Total generated turns: 165
   conversation_id  turn_number  \
0               68            6   
1               68            7   
2               68            8   
3               68            9   
4               68           10   
5               72            6   
6               72            7   
7               72            8   
8               72            9   
9               72           10   

                                  generated_response  
0  I would do anything to protect my family, but ...  
1  good talking to you. I hope it works out for t...  
2  Going back to what you said earlier. I feel ba...  
3  Yeah! I hope the families that went through th...  
4  I just hope that the organizations are able to...  
5  This article was very eye opening. I had no id...  
6  What do you think about the people dying in In...  
7  I wish that article talked about the effect of...  
8  i was thinking the same thing,i wonder how

In [8]:
!pip install evaluate
import evaluate

# Load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')

# Get gold responses for dev set (turns 6-10)
gold_responses = []
pred_responses = []

for conv_id in dev_df['conversation_id'].unique():
    conv = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')
    gold_turns = conv[conv['turn_id'] >= 6].head(5)['text'].astype(str).tolist()
    pred_turns = dev_results_df[dev_results_df['conversation_id'] == conv_id]['generated_response'].tolist()

    min_len = min(len(gold_turns), len(pred_turns))
    gold_responses.extend(gold_turns[:min_len])
    pred_responses.extend(pred_turns[:min_len])

# ROUGE
rouge_scores = rouge.compute(predictions=pred_responses, references=gold_responses)
print("=== ROUGE ===")
print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")

# BLEU
bleu_score = bleu.compute(predictions=pred_responses, references=[[g] for g in gold_responses])
print("\n=== BLEU ===")
print(f"BLEU: {bleu_score['bleu']:.4f}")

# BertScore
print("\nComputing BertScore (this may take a minute)...")
bert_scores = bertscore.compute(predictions=pred_responses, references=gold_responses, lang='en')
print("=== BertScore ===")
print(f"Precision: {sum(bert_scores['precision'])/len(bert_scores['precision']):.4f}")
print(f"Recall: {sum(bert_scores['recall'])/len(bert_scores['recall']):.4f}")
print(f"F1: {sum(bert_scores['f1'])/len(bert_scores['f1']):.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


=== ROUGE ===
ROUGE-1: 0.1279
ROUGE-2: 0.0162
ROUGE-L: 0.1037

=== BLEU ===
BLEU: 0.0000

Computing BertScore (this may take a minute)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== BertScore ===
Precision: 0.8619
Recall: 0.8548
F1: 0.8582


In [9]:
test_p2_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/refs/heads/main/P2%20Test%20Data/project_part2_test.csv', on_bad_lines='skip')
print(test_p2_df.shape)
print(test_p2_df.columns.tolist())
print(test_p2_df.head(3))

(2316, 11)
['id', 'article_id', 'conversation_id', 'turn_id', 'speaker_id', 'text', 'Emotion', 'EmotionalPolarity', 'Empathy', 'Unnamed: 9', 'Unnamed: 10']
   id  article_id  conversation_id  turn_id  speaker_id  \
0   1          27              393        1         188   
1   2          27              393        2         189   
2   3          27              393        3         188   

                                                text   Emotion  \
0  Yeah, I'm sorry but celebrity life doesn't int...  1.333333   
1  I'm pretty much the same. I like some of the m...  1.333333   
2  Right? And I think that some of their problems...  1.666667   

   EmotionalPolarity  Empathy  Unnamed: 9  Unnamed: 10  
0                1.0      1.0         NaN          NaN  
1                0.5      1.0         NaN          NaN  
2                1.0      1.0         NaN          NaN  


In [10]:
print("Generating for test set...")
test_results = []

for conv_id in test_p2_df['conversation_id'].unique():
    conv = test_p2_df[test_p2_df['conversation_id'] == conv_id].sort_values('turn_id')

    # Get first 5 turns as history
    history = conv[conv['turn_id'] <= 5]['text'].astype(str).tolist()
    if len(history) == 0:
        continue

    # Get target emotion/empathy/polarity for turns 6-10
    emotion_targets, empathy_targets, polarity_targets = get_target_values(test_p2_df, conv_id)

    # Generate next 5 turns
    generated = generate_conversation(
        history.copy(),
        emotion_targets,
        empathy_targets,
        polarity_targets,
        train_embeddings,
        train_df,
        sbert_model
    )

    for turn_num, utterance in enumerate(generated, start=6):
        test_results.append({
            'conversation_id': conv_id,
            'turn_number': turn_num,
            'generated_response': utterance
        })

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv('generations_corpus.csv', index=False)
print("Saved generations_corpus.csv!")
print(f"Total generated turns: {len(test_results_df)}")
print(test_results_df.head(10))

Generating for test set...
Saved generations_corpus.csv!
Total generated turns: 335
   conversation_id  turn_number  \
0              393            6   
1              393            7   
2              393            8   
3              393            9   
4              393           10   
5              394            6   
6              394            7   
7              394            8   
8              394            9   
9              394           10   

                                  generated_response  
0  Ehh I wonder. Sometimes it seems that certain ...  
1  Same here. I don't know why you'd ever enjoy h...  
2  yeah, i never hear anything about this. All yo...  
3  Maybe I guess I just can't wrap my head around...  
4  yeah me and you can both be journalists nowada...  
5  I agree and she was so young when they married...  
6  That's true. Are you a fan of Billy BOb Thornt...  
7  I wonder how Dakota and Elle Fanning are doing...  
8  Yeah.  Thing is though, Lebron w

In [11]:
print("Generating 5 conversations with 10 utterances each...")
long_results = []

# Pick 5 conversation ids from dev set
sample_conv_ids = dev_df['conversation_id'].unique()[:5]

for conv_id in sample_conv_ids:
    conv = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')

    # Get first 5 turns as history
    history = conv[conv['turn_id'] <= 5]['text'].astype(str).tolist()
    if len(history) == 0:
        continue

    # Get target values for 10 turns
    emotion_targets, empathy_targets, polarity_targets = get_target_values(dev_df, conv_id, turn_start=6, n_turns=10)

    # Generate next 10 turns
    generated = generate_conversation(
        history.copy(),
        emotion_targets,
        empathy_targets,
        polarity_targets,
        train_embeddings,
        train_df,
        sbert_model,
        n_turns=10
    )

    for turn_num, utterance in enumerate(generated, start=6):
        long_results.append({
            'conversation_id': conv_id,
            'turn_number': turn_num,
            'generated_response': utterance
        })

long_results_df = pd.DataFrame(long_results)
long_results_df.to_csv('generations_corpus_10turns.csv', index=False)
print("Saved generations_corpus_10turns.csv!")
print(long_results_df.to_string())

Generating 5 conversations with 10 utterances each...
Saved generations_corpus_10turns.csv!
    conversation_id  turn_number                                                                                                                                                                                           generated_response
0                68            6                                                I would do anything to protect my family, but I do not  go looking for trouble. I'm glad you can see it from an objective point of view because most can not.
1                68            7                                                                                                                                       good talking to you. I hope it works out for the people affected. bye.
2                68            8                                                                                                               Going back to what you said earlier. I feel bad for